# Everything, end to end

Runs every job in the catalog, group by group, through Kyuubi, and ends with a pass/fail
table. This is `make test` from a notebook. Kafka jobs are skipped when Kafka isn't
installed; the JVM jobs are skipped until you `make jars && make upload`. Takes 20-40
minutes on a laptop.

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
import time, job_catalog, lab_runner
GROUPS = job_catalog.GROUPS          # or e.g. ["core", "formats", "hive", "postgres"]
rows, failed = [], set()
t0 = time.time()
for g in GROUPS:
    for c in [c for c in job_catalog.CASES if c["group"] == g]:
        if set(c.get("after") or []) & failed:
            rows.append(dict(group=g, name=c["name"], result="skipped (input failed)"))
            failed.add(c["name"]); continue
        try:
            r = lab.run(c["name"], run_id=RID)
            res = "skipped: " + r["detail"] if r["ok"] is None else "pass"
            if r["ok"] is None: failed.add(c["name"])
            rows.append(dict(group=g, name=c["name"], result=res, secs=r.get("secs"), batch=r.get("batch_id")))
        except Exception as e:
            failed.add(c["name"])
            rows.append(dict(group=g, name=c["name"], result="FAIL", detail=str(e)[:120]))
print(f"done in {int(time.time() - t0)}s")
report = pd.DataFrame(rows)
report

In [ ]:
report["status"] = report["result"].str.split(":").str[0].str.split(" ").str[0]
report.groupby(["group", "status"]).size().unstack(fill_value=0)

In [ ]:
ax = report.groupby(["group", "status"]).size().unstack(fill_value=0).plot.bar(stacked=True, title="end-to-end results")

In [ ]:
bad = report[report["result"] == "FAIL"]
assert bad.empty, f"{len(bad)} job(s) failed:\n{bad[['name', 'detail']].to_string()}"
print("all jobs passed (or were skipped for a missing optional service)")